# Medical Diagnosis AI — Sequential NHS Inform Scraper

This notebook scrapes NHS Inform **one condition at a time automatically**, but you only press **Run once** for the main crawl.

The algorithm does this:

```text
Condition 1
   ↓
scrape
   ↓
save immediately
   ↓
wait
   ↓
Condition 2
   ↓
scrape
   ↓
save immediately
   ↓
wait
   ↓
...
```

It also includes a **warning-text fallback**.

If a page has no dedicated warning heading but contains urgent text such as:

- `phone 999`
- `phone 111`
- `contact your GP urgently`
- `seek immediate medical help`
- `go to A&E`

the scraper can still capture that content into `warnings`.

Nothing is written to MongoDB yet.


## Step 1 — Install Requirements

Run once if needed:

```bash
pip install requests beautifulsoup4 pandas
```


## Step 2 — Imports

In [1]:
import sys
import json
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse, urldefrag

import requests
import pandas as pd
from bs4 import BeautifulSoup

try:
    sys.stdout.reconfigure(encoding="utf-8")
except AttributeError:
    pass


## Step 3 — Configuration

The crawler processes one condition, saves it, waits, then moves to the next.

You do **not** manually run each condition.


In [2]:
A_Z_URL = (
    "https://www.nhsinform.scot/"
    "illnesses-and-conditions/a-to-z/"
)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/154.0.0.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,application/xml;"
        "q=0.9,image/avif,image/webp,*/*;q=0.8"
    ),
    "Accept-Language": "en-GB,en;q=0.9",
    "Connection": "keep-alive",
}

REQUEST_TIMEOUT = 25

# Wait between conditions to reduce the chance of blocking.
REQUEST_DELAY_SECONDS = 5

MAX_RETRIES = 3
BACKOFF_BASE_SECONDS = 10

OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(exist_ok=True)

RESULTS_JSON = OUTPUT_DIR / "nhs_conditions_sequential.json"
RESULTS_CSV = OUTPUT_DIR / "nhs_conditions_sequential.csv"

ERRORS_JSON = OUTPUT_DIR / "nhs_conditions_errors.json"
ERRORS_CSV = OUTPUT_DIR / "nhs_conditions_errors.csv"


## Step 4 — Section Keywords

These keywords map NHS headings into your required dataset fields.


In [3]:
SECTION_KEYWORDS = {
    "symptoms": [
        "symptom",
        "signs",
    ],

    "causes": [
        "cause",
        "trigger",
        "risk factor",
    ],

    "warnings": [
        "when to get medical advice",
        "when to get medical help",
        "when to seek medical advice",
        "when to seek help",
        "urgent",
        "emergency",
        "call 999",
        "phone 999",
        "a&e",
    ],

    "recommendations": [
        "treatment",
        "treating",
        "how is",
        "how to treat",
        "self-help",
        "self care",
        "self-care",
        "management",
        "managing",
        "prevention",
        "preventing",
        "living with",
        "what to do",
        "things you can do",
    ],
}

RECOMMENDATION_LINK_KEYWORDS = [
    "treatment",
    "treatments",
    "prevention",
    "preventing",
    "management",
    "managing",
]

WARNING_TEXT_KEYWORDS = [
    "phone 999",
    "call 999",
    "dial 999",
    "phone 111",
    "call 111",
    "nhs 24 111",
    "contact your gp urgently",
    "contact your gp immediately",
    "contact your gp",
    "seek urgent medical help",
    "seek immediate medical help",
    "get urgent medical help",
    "get immediate medical help",
    "go to a&e",
    "go to accident and emergency",
    "go to hospital immediately",
    "emergency department",
    "ambulance",
]


## Step 5 — General Helpers

In [4]:
def clean_text(element):
    if not element:
        return ""
    return " ".join(element.stripped_strings)


def detect_category(heading_text):
    heading_text = heading_text.lower()

    for category, keywords in SECTION_KEYWORDS.items():
        for keyword in keywords:
            if keyword in heading_text:
                return category

    return None


def normalize_url(url):
    clean_url, _ = urldefrag(url)
    return clean_url.rstrip("/") + "/"


## Step 6 — Request One Page with Retry + Backoff

Each condition is requested separately.

Temporary HTTP errors such as `403`, `429`, or server errors are retried before moving on.


In [5]:
RETRY_STATUS_CODES = {
    403,
    408,
    429,
    500,
    502,
    503,
    504,
}


def get_with_retry(
    session,
    url,
    max_retries=MAX_RETRIES,
):
    for attempt in range(max_retries + 1):

        try:
            response = session.get(
                url,
                timeout=REQUEST_TIMEOUT
            )

            if response.status_code == 200:
                return response

            if response.status_code not in RETRY_STATUS_CODES:
                response.raise_for_status()

            if attempt >= max_retries:
                response.raise_for_status()

            wait_seconds = (
                BACKOFF_BASE_SECONDS
                * (2 ** attempt)
            )

            print(
                f"    HTTP {response.status_code}. "
                f"Retrying in {wait_seconds}s..."
            )

            time.sleep(wait_seconds)

        except requests.RequestException:
            if attempt >= max_retries:
                raise

            wait_seconds = (
                BACKOFF_BASE_SECONDS
                * (2 ** attempt)
            )

            print(
                f"    Request failed. "
                f"Retrying in {wait_seconds}s..."
            )

            time.sleep(wait_seconds)

    raise RuntimeError(
        f"Failed to request: {url}"
    )


## Step 7 — Main Medical Section Extraction

In [6]:
def extract_medical_sections(soup):
    result = {
        "symptoms": [],
        "causes": [],
        "warnings": [],
        "recommendations": [],
    }

    current_h2_category = None
    active_category = None

    elements = soup.find_all(
        ["h2", "h3", "p", "ul", "ol"]
    )

    for element in elements:

        if element.name == "h2":
            heading = clean_text(element)

            if not heading:
                continue

            current_h2_category = detect_category(heading)
            active_category = current_h2_category
            continue

        if element.name == "h3":
            heading = clean_text(element)

            if not heading:
                continue

            own_category = detect_category(heading)

            if own_category:
                active_category = own_category
            else:
                active_category = current_h2_category

            continue

        if active_category:
            text = clean_text(element)

            if text:
                result[active_category].append(text)

    for category in result:
        result[category] = " ".join(
            result[category]
        )

    return result


## Step 8 — Symptoms Intro Fallback

Some NHS pages put symptom information before the first `H2`.

If a dedicated symptoms section is missing, this intro content becomes the fallback.


In [7]:
def extract_intro_content(soup):
    title = soup.find("h1")

    if not title:
        return ""

    intro_parts = []

    for element in title.find_all_next(
        ["p", "ul", "ol", "h2"]
    ):
        if element.name == "h2":
            break

        text = clean_text(element)

        if text:
            intro_parts.append(text)

    return " ".join(intro_parts)


## Step 9 — Warning Text Fallback

Some pages contain urgent instructions inside ordinary paragraphs rather than a dedicated warning heading.

This scans paragraphs and lists for urgent-action phrases.

It is only used when the normal `warnings` extraction is empty.


In [8]:
def extract_warning_text_fallback(soup):
    warning_parts = []
    seen = set()

    for element in soup.find_all(
        ["p", "ul", "ol"]
    ):
        text = clean_text(element)

        if not text:
            continue

        text_lower = text.lower()

        has_warning_keyword = any(
            keyword in text_lower
            for keyword in WARNING_TEXT_KEYWORDS
        )

        if not has_warning_keyword:
            continue

        # Avoid duplicate text caused by nested elements.
        normalized_text = " ".join(
            text.split()
        )

        if normalized_text in seen:
            continue

        seen.add(normalized_text)
        warning_parts.append(
            normalized_text
        )

    return " ".join(
        warning_parts
    )


## Step 10 — Linked Treatment / Prevention Fallback


In [9]:
def find_recommendation_links(
    soup,
    base_url,
    condition_name
):
    links = []

    base_domain = urlparse(base_url).netloc.lower()

    condition_words = [
        word.lower()
        for word in condition_name.split()
        if len(word) > 3
    ]

    for anchor in soup.find_all(
        "a",
        href=True
    ):
        link_text = clean_text(anchor).lower()

        absolute_url = normalize_url(
            urljoin(
                base_url,
                anchor.get("href", "")
            )
        )

        parsed = urlparse(absolute_url)

        if parsed.netloc.lower() != base_domain:
            continue

        url_lower = absolute_url.lower()

        if "/illnesses-and-conditions/" not in url_lower:
            continue

        combined = link_text + " " + url_lower

        if not any(
            keyword in combined
            for keyword in RECOMMENDATION_LINK_KEYWORDS
        ):
            continue

        if not any(
            word in combined
            for word in condition_words
        ):
            continue

        if absolute_url.rstrip("/") == base_url.rstrip("/"):
            continue

        if absolute_url not in links:
            links.append(absolute_url)

    return links


def scrape_linked_recommendations(
    session,
    links,
    original_url,
):
    for link in links:

        if link.rstrip("/") == original_url.rstrip("/"):
            continue

        try:
            response = get_with_retry(
                session,
                link
            )

            linked_soup = BeautifulSoup(
                response.content,
                "html.parser"
            )

            sections = extract_medical_sections(
                linked_soup
            )

            recommendations = sections[
                "recommendations"
            ]

            if recommendations:
                return (
                    recommendations,
                    normalize_url(link)
                )

        except Exception:
            continue

    return "", None


## Step 11 — Scrape Exactly One Condition

This function handles a single condition page.

The outer loop later calls this function one condition at a time automatically.


In [10]:
def scrape_condition(
    url,
    session
):
    response = get_with_retry(
        session,
        url
    )

    soup = BeautifulSoup(
        response.content,
        "html.parser"
    )

    title = soup.find("h1")

    if not title:
        raise ValueError(
            "Could not find condition title"
        )

    condition_name = clean_text(title)

    sections = extract_medical_sections(
        soup
    )

    symptoms = sections["symptoms"]
    causes = sections["causes"]
    warnings = sections["warnings"]
    recommendations = sections["recommendations"]

    symptoms_source = "section"
    warnings_source = "section"
    recommendations_source = "section"

    # Symptoms fallback
    if not symptoms:
        intro_content = extract_intro_content(
            soup
        )

        if intro_content:
            symptoms = intro_content
            symptoms_source = "intro_fallback"

    # Warning fallback
    if not warnings:
        warning_fallback = (
            extract_warning_text_fallback(
                soup
            )
        )

        if warning_fallback:
            warnings = warning_fallback
            warnings_source = "text_fallback"
        else:
            warnings_source = "not_found"

    # Recommendations fallback
    linked_recommendation_url = None

    if not recommendations:
        recommendation_links = (
            find_recommendation_links(
                soup,
                response.url,
                condition_name
            )
        )

        (
            recommendations,
            linked_recommendation_url
        ) = scrape_linked_recommendations(
            session,
            recommendation_links,
            response.url,
        )

        if recommendations:
            recommendations_source = (
                "linked_page_fallback"
            )
        else:
            recommendations_source = (
                "not_found"
            )

    return {
        "condition": condition_name,
        "symptoms": symptoms,
        "causes": causes,
        "warnings": warnings,
        "recommendations": recommendations,
        "source_url": normalize_url(
            response.url
        ),
        "symptoms_source": symptoms_source,
        "warnings_source": warnings_source,
        "recommendations_source": recommendations_source,
        "recommendations_source_url": linked_recommendation_url,
    }


## Step 12 — Discover the NHS Inform A–Z Condition List

This reads the A–Z index once and builds the list of condition URLs.


In [11]:
EXCLUDED_INDEX_LABELS = {
    "illnesses and conditions",
}

EXCLUDED_LABEL_PREFIXES = [
    "how to ",
    "living with ",
    "living well with ",
    "managing ",
]


def is_probable_condition_entry(
    label,
    url
):
    label_clean = label.strip().lower()
    url_clean = normalize_url(url).lower()

    if not label_clean:
        return False

    if label_clean in EXCLUDED_INDEX_LABELS:
        return False

    for prefix in EXCLUDED_LABEL_PREFIXES:
        if label_clean.startswith(prefix):
            return False

    if "/a-to-z/" in url_clean:
        return False

    root_url = normalize_url(
        "https://www.nhsinform.scot/"
        "illnesses-and-conditions/"
    ).lower()

    if url_clean == root_url:
        return False

    if "/illnesses-and-conditions/" not in url_clean:
        return False

    return True


def get_condition_links():
    session = requests.Session()
    session.headers.update(HEADERS)

    response = get_with_retry(
        session,
        A_Z_URL
    )

    soup = BeautifulSoup(
        response.content,
        "html.parser"
    )

    discovered = {}

    for anchor in soup.find_all(
        "a",
        href=True
    ):
        label = clean_text(anchor)

        absolute_url = normalize_url(
            urljoin(
                A_Z_URL,
                anchor.get("href", "")
            )
        )

        if not is_probable_condition_entry(
            label,
            absolute_url
        ):
            continue

        discovered[absolute_url] = label

    return [
        {
            "index_label": label,
            "url": url,
        }
        for url, label in sorted(
            discovered.items(),
            key=lambda item: item[1].lower()
        )
    ]


condition_links = get_condition_links()

print(
    "Conditions discovered:",
    len(condition_links)
)

pd.DataFrame(
    condition_links
).head(20)


Conditions discovered: 447


,index_label,url
0,About aplastic anaemia,https://www.nhsinform.scot/illnesses-and-condi...
1,About scoliosis,https://www.nhsinform.scot/illnesses-and-condi...
2,Achilles tendinopathy,https://www.nhsinform.scot/illnesses-and-condi...
3,Acid and chemical burns,https://www.nhsinform.scot/illnesses-and-condi...
4,Acne,https://www.nhsinform.scot/illnesses-and-condi...
5,Acute cholecystitis,https://www.nhsinform.scot/illnesses-and-condi...
6,Acute lymphoblastic leukaemia,https://www.nhsinform.scot/illnesses-and-condi...
7,Acute myeloid leukaemia,https://www.nhsinform.scot/illnesses-and-condi...
8,Acute pancreatitis,https://www.nhsinform.scot/illnesses-and-condi...
9,Acute respiratory infection (ARI),https://www.nhsinform.scot/illnesses-and-condi...


## Step 13 — Save Progress After Every Condition

Every successful condition is saved immediately before the next condition starts.


In [12]:
def load_json(
    path,
    default=None
):
    if default is None:
        default = []

    if not path.exists():
        return default

    try:
        with path.open(
            "r",
            encoding="utf-8"
        ) as file:
            return json.load(file)

    except Exception:
        return default


def save_results(results):
    with RESULTS_JSON.open(
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            results,
            file,
            ensure_ascii=False,
            indent=2
        )

    pd.DataFrame(
        results
    ).to_csv(
        RESULTS_CSV,
        index=False,
        encoding="utf-8-sig"
    )


def save_errors(errors):
    with ERRORS_JSON.open(
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            errors,
            file,
            ensure_ascii=False,
            indent=2
        )

    pd.DataFrame(
        errors
    ).to_csv(
        ERRORS_CSV,
        index=False,
        encoding="utf-8-sig"
    )


# Step 14 — Run the Full Sequential Scrape

## This is the only main crawl cell you need to run once

It automatically does:

```text
scrape one
→ save one
→ wait
→ scrape next
→ save next
→ wait
→ ...
```

If you stop the notebook and rerun later, already-saved URLs are skipped.


In [ ]:
results = load_json(
    RESULTS_JSON,
    []
)

errors = load_json(
    ERRORS_JSON,
    []
)

saved_urls = {
    normalize_url(
        record["source_url"]
    )
    for record in results
    if record.get("source_url")
}

session = requests.Session()
session.headers.update(HEADERS)

total = len(condition_links)

print(
    f"Already saved: {len(results)}"
)

print(
    f"Total conditions: {total}"
)

print(
    f"Remaining: {total - len(saved_urls)}"
)

print()


for index, item in enumerate(
    condition_links,
    start=1
):
    target_url = normalize_url(
        item["url"]
    )

    if target_url in saved_urls:
        print(
            f"[{index}/{total}] "
            f"SKIP → {item['index_label']}"
        )
        continue

    print(
        f"[{index}/{total}] "
        f"SCRAPING → {item['index_label']}"
    )

    try:
        result = scrape_condition(
            item["url"],
            session
        )

        result["index_label"] = (
            item["index_label"]
        )

        results.append(
            result
        )

        saved_urls.add(
            normalize_url(
                result["source_url"]
            )
        )

        # Remove a previous error for this URL
        # if the page succeeds on a later run.
        errors = [
            error
            for error in errors
            if normalize_url(
                error.get(
                    "url",
                    ""
                )
            ) != target_url
        ]

        # Save immediately before moving on.
        save_results(results)
        save_errors(errors)

        print(
            f"    SAVED → "
            f"{result['condition']}"
        )

        print(
            "    "
            f"symptoms={len(result['symptoms'])}, "
            f"causes={len(result['causes'])}, "
            f"warnings={len(result['warnings'])}, "
            f"recommendations={len(result['recommendations'])}"
        )

        print(
            "    "
            f"warnings_source="
            f"{result['warnings_source']}"
        )

    except Exception as error:
        print(
            f"    FAILED → {error}"
        )

        errors = [
            existing
            for existing in errors
            if normalize_url(
                existing.get(
                    "url",
                    ""
                )
            ) != target_url
        ]

        errors.append({
            "index_label":
                item["index_label"],
            "url":
                item["url"],
            "error":
                str(error),
        })

        save_errors(errors)

    if index < total:
        print(
            f"    Waiting "
            f"{REQUEST_DELAY_SECONDS}s "
            f"before next condition..."
        )

        time.sleep(
            REQUEST_DELAY_SECONDS
        )

print()
print("=" * 70)
print("SEQUENTIAL SCRAPE FINISHED")
print("=" * 70)

print(
    "Successful records:",
    len(results)
)

print(
    "Remaining errors:",
    len(errors)
)


Already saved: 0
Total conditions: 447
Remaining: 447

[1/447] SCRAPING → About aplastic anaemia
    SAVED → About aplastic anaemia
    symptoms=471, causes=737, warnings=0, recommendations=1411
    warnings_source=not_found
    Waiting 5s before next condition...
[2/447] SCRAPING → About scoliosis
    SAVED → About scoliosis
    symptoms=375, causes=1089, warnings=0, recommendations=102
    warnings_source=not_found
    Waiting 5s before next condition...
[3/447] SCRAPING → Achilles tendinopathy
    SAVED → Achilles tendinopathy
    symptoms=376, causes=603, warnings=0, recommendations=557
    warnings_source=not_found
    Waiting 5s before next condition...
[4/447] SCRAPING → Acid and chemical burns
    SAVED → Acid and chemical burns
    symptoms=145, causes=0, warnings=163, recommendations=250
    warnings_source=text_fallback
    Waiting 5s before next condition...
[5/447] SCRAPING → Acne
    SAVED → Acne
    symptoms=515, causes=1150, warnings=0, recommendations=2277
    warnings

## Step 15 — Inspect Generated Dataset

In [ ]:
results = load_json(
    RESULTS_JSON,
    []
)

results_df = pd.DataFrame(
    results
)

print(
    "Saved records:",
    len(results_df)
)

results_df.head()


## Step 16 — Review Errors

In [ ]:
errors = load_json(
    ERRORS_JSON,
    []
)

errors_df = pd.DataFrame(
    errors
)

print(
    "Failed conditions:",
    len(errors_df)
)

errors_df


## Step 17 — Field Coverage

This shows how many records contain each field.

An empty warning field can still be valid if NHS Inform contains no warning guidance for that page.


In [ ]:
if not results_df.empty:

    coverage = pd.DataFrame({
        "field": [
            "symptoms",
            "causes",
            "warnings",
            "recommendations",
        ],

        "records_with_content": [
            results_df[
                "symptoms"
            ].fillna("").str.strip().ne("").sum(),

            results_df[
                "causes"
            ].fillna("").str.strip().ne("").sum(),

            results_df[
                "warnings"
            ].fillna("").str.strip().ne("").sum(),

            results_df[
                "recommendations"
            ].fillna("").str.strip().ne("").sum(),
        ],
    })

    coverage[
        "total_records"
    ] = len(results_df)

    coverage


## Step 18 — Check Warning Extraction Sources

This lets you see how many warnings came from:

- normal section headings
- text fallback
- no warning found


In [ ]:
if (
    not results_df.empty
    and "warnings_source"
    in results_df.columns
):
    results_df[
        "warnings_source"
    ].value_counts(
        dropna=False
    )


## Step 19 — Inspect One Condition Manually


In [ ]:
CONDITION_TO_INSPECT = "Asthma"

matches = [
    record
    for record in results
    if record.get(
        "condition",
        ""
    ).lower()
    == CONDITION_TO_INSPECT.lower()
]

if not matches:
    print(
        "Condition not found."
    )

else:
    record = matches[0]

    print(
        "CONDITION:",
        record["condition"]
    )

    print(
        "WARNINGS SOURCE:",
        record.get(
            "warnings_source"
        )
    )

    print(
        "\n--- SYMPTOMS ---\n"
    )
    print(
        record["symptoms"]
        or "[EMPTY]"
    )

    print(
        "\n--- CAUSES ---\n"
    )
    print(
        record["causes"]
        or "[EMPTY]"
    )

    print(
        "\n--- WARNINGS ---\n"
    )
    print(
        record["warnings"]
        or "[EMPTY]"
    )

    print(
        "\n--- RECOMMENDATIONS ---\n"
    )
    print(
        record["recommendations"]
        or "[EMPTY]"
    )


# Output Files

The single sequential run continuously builds:

```text
data/
├── nhs_conditions_sequential.json
├── nhs_conditions_sequential.csv
├── nhs_conditions_errors.json
└── nhs_conditions_errors.csv
```

Every condition is handled separately, but the **loop advances automatically**.

You press Run once on Step 14 and let it continue.

If it stops, rerun the notebook later and Step 14 will skip conditions already saved.
